# MedSAM2 Video Evaluation — Zero-Shot vs Fine-Tuned MedSAM2
**Dataset:** CoronaryDominance (10 studies, vid1–vid10)  
**Comparison:** Original | Zero-Shot MedSAM2 | Fine-Tuned MedSAM2  
**Prompt:** Single centroid click on auto-selected prompt frame

## 1 — Setup

In [ ]:
import os, subprocess, sys

subprocess.run(['sudo','apt-get','update','-q'], check=True)
subprocess.run(['sudo','apt-get','install','-y','ffmpeg','p7zip-full'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'torch==2.5.1','torchvision==0.20.1',
    '--index-url','https://download.pytorch.org/whl/cu121','-q'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'numpy>=2.0.1','opencv-python','pillow','matplotlib','tqdm',
    'hydra-core','omegaconf','iopath','-q'], check=True)

if not os.path.exists('/home/jupyter/MedSAM2'):
    subprocess.run(['git','clone','https://github.com/bowang-lab/MedSAM2.git',
                    '/home/jupyter/MedSAM2'], check=True)

sys.path.insert(0, '/home/jupyter/MedSAM2')
print('Environment ready')

## 2 — Config & Paths

In [ ]:
import os

BUCKET      = 'gs://YOUR_GCS_BUCKET'  # <-- set your bucket
CONFIG      = 'configs/sam2.1_hiera_t512.yaml'
ZS_CKPT     = '/home/jupyter/MedSAM2/checkpoints/MedSAM2_latest.pt'
FT_CKPT     = '/home/jupyter/medsam2_arcade_v2.pt'
CD_DIR      = '/tmp/coronary_dominance'
N_STUDIES   = 10
VID_FRAMES  = [f'/home/jupyter/vid{i+1}_frames' for i in range(N_STUDIES)]
OUT_DIR     = '/home/jupyter/video_eval'

for d in [CD_DIR, OUT_DIR] + VID_FRAMES:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(FT_CKPT):
    print('Pulling ARCADE v2 checkpoint from GCS...')
    subprocess.run(['gsutil','cp',
        f'{BUCKET}/checkpoints/medsam2_arcade_v2.pt', FT_CKPT], check=True)

if not os.path.exists(ZS_CKPT):
    print('Pulling MedSAM2_latest.pt from GCS...')
    subprocess.run(['gsutil','cp',
        f'{BUCKET}/checkpoints/MedSAM2_latest.pt', ZS_CKPT], check=True)

print(f'Zero-shot : {ZS_CKPT}  exists={os.path.exists(ZS_CKPT)}')
print(f'Fine-tuned: {FT_CKPT}  exists={os.path.exists(FT_CKPT)}')
print(f'Running {N_STUDIES} studies → vid1–vid{N_STUDIES}')

## 3 — Pull & Extract CoronaryDominance Data

In [ ]:
import glob, re
from collections import defaultdict

if not glob.glob(f'{CD_DIR}/**/*.npz', recursive=True):
    print('Downloading domain_shift_dataset.7z from GCS (~7.8 GB)...')
    subprocess.run(['gsutil','cp',
        f'{BUCKET}/domain_shift_dataset.7z',
        '/tmp/domain_shift_dataset.7z'], check=True)
    print('Extracting...')
    subprocess.run(['7z','x','/tmp/domain_shift_dataset.7z',
        f'-o{CD_DIR}/','-y'], check=True)
    os.remove('/tmp/domain_shift_dataset.7z')
    print('Done')
else:
    print('Data already on disk')

npz_all = sorted(glob.glob(f'{CD_DIR}/**/*.npz', recursive=True))
print(f'Total NPZ files: {len(npz_all)}')

study_map = defaultdict(list)
for f in npz_all:
    m = re.search(r'(Study\w+)', f)
    study_id = m.group(1) if m else os.path.basename(os.path.dirname(os.path.dirname(f)))
    study_map[study_id].append(f)

studies   = sorted(study_map.keys())[:N_STUDIES]
STUDY_NPZS = [max(study_map[s], key=os.path.getsize) for s in studies]

print(f'\nSelected {len(STUDY_NPZS)} studies:')
for i, npz in enumerate(STUDY_NPZS):
    print(f'  vid{i+1}: {os.path.basename(npz)}  ({os.path.getsize(npz)/1e6:.0f} MB)')

## 4 — Extract Frames & Inspect

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def extract_frames(npz_path, frames_dir):
    existing = sorted(glob.glob(f'{frames_dir}/*.jpeg'))
    if existing:
        print(f'  {len(existing)} frames already extracted')
        return existing

    data = np.load(npz_path, allow_pickle=True)
    frame_array = None
    for key in ['imgs', 'frames', 'images', 'data', 'video']:
        if key in data:
            frame_array = data[key]
            break
    if frame_array is None:
        frame_array = list(data.values())[0]

    saved = []
    for i, frame in enumerate(frame_array):
        if frame.max() <= 1.0:
            frame = (frame * 255).astype(np.uint8)
        else:
            frame = frame.astype(np.uint8)
        if frame.ndim == 2:
            frame = np.stack([frame]*3, axis=-1)
        elif frame.ndim == 3 and frame.shape[0] in [1, 3]:
            frame = np.transpose(frame, (1, 2, 0))
            if frame.shape[2] == 1:
                frame = np.repeat(frame, 3, axis=2)
        path = f'{frames_dir}/{i:05d}.jpeg'
        Image.fromarray(frame).save(path, quality=95)
        saved.append(path)
    return saved

all_vid_frames = []
for i, (npz, frames_dir) in enumerate(zip(STUDY_NPZS, VID_FRAMES)):
    print(f'vid{i+1}:', end=' ')
    frames = extract_frames(npz, frames_dir)
    print(f'{len(frames)} frames')
    all_vid_frames.append(frames)

# 2×5 grid of first frames
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for i, (frames, ax) in enumerate(zip(all_vid_frames, axes.flat)):
    ax.imshow(np.array(Image.open(frames[0])), cmap='gray')
    ax.set_title(f'vid{i+1}  ({len(frames)} frames)', fontsize=9)
    ax.axis('off')
plt.suptitle('First frame — all 10 studies', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/all_first_frames.png', dpi=110)
plt.show()

## 5 — Auto-Select Prompt Frame & Centroid Click

In [ ]:
def auto_prompt(frame_files):
    variances = [np.array(Image.open(f).convert('L'), dtype=np.float32).var()
                 for f in frame_files]
    pf   = int(np.argmax(variances))
    gray = np.array(Image.open(frame_files[pf]).convert('L'), dtype=np.float32)
    H, W = gray.shape
    ys, xs = np.mgrid[0:H, 0:W]
    sigma  = min(H, W) / 3.0
    cw     = np.exp(-((xs - W/2)**2 + (ys - H/2)**2) / (2 * sigma**2))
    weighted = gray * cw
    thresh   = np.percentile(weighted[weighted > 0], 85)
    by, bx   = np.where(weighted > thresh)
    return pf, int(np.mean(bx)), int(np.mean(by))

prompts = [auto_prompt(frames) for frames in all_vid_frames]

# 2×5 grid showing prompt frame + click for all 10 studies
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for i, (frames, (pf, cx, cy), ax) in enumerate(zip(all_vid_frames, prompts, axes.flat)):
    ax.imshow(np.array(Image.open(frames[pf])), cmap='gray')
    ax.plot(cx, cy, 'r+', markersize=18, markeredgewidth=2)
    ax.set_title(f'vid{i+1} — Fr.{pf}  click({cx},{cy})', fontsize=8)
    ax.axis('off')
plt.suptitle('Auto-selected prompt frames and centroid clicks — all 10 studies', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/all_prompt_frames.png', dpi=110)
plt.show()

for i, (pf, cx, cy) in enumerate(prompts):
    print(f'vid{i+1}: prompt_frame={pf}  click=({cx},{cy})')

## 6 — Load Models

In [ ]:
import torch
from sam2.build_sam import build_sam2_video_predictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "n/a"}')

# Zero-shot: original MedSAM2 weights, no fine-tuning
print('\nLoading zero-shot predictor (original MedSAM2)...')
zs_predictor = build_sam2_video_predictor(CONFIG, ZS_CKPT, device=device)
print('Zero-shot ready')

# Fine-tuned: same architecture, ARCADE v2 weights injected
print('\nLoading fine-tuned predictor (ARCADE v2)...')
ft_predictor = build_sam2_video_predictor(CONFIG, ZS_CKPT, device=device)
ft_state = torch.load(FT_CKPT, map_location=device, weights_only=False)
missing, unexpected = ft_predictor.load_state_dict(ft_state, strict=False)
ft_predictor.eval()
print(f'ARCADE v2 ready  |  missing={len(missing)}  unexpected={len(unexpected)}')

## 7 — Run Inference (Both Models, Both Studies)

In [ ]:
def run_video_inference(predictor, frames_dir, prompt_frame, click_x, click_y):
    masks = {}
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        state = predictor.init_state(video_path=frames_dir)
        predictor.add_new_points_or_box(
            inference_state=state,
            frame_idx=prompt_frame,
            obj_id=1,
            points=np.array([[click_x, click_y]], dtype=np.float32),
            labels=np.array([1], dtype=np.int32),
        )
        for out_idx, _, out_logits in predictor.propagate_in_video(state):
            masks[out_idx] = (out_logits[0, 0] > 0.0).cpu().numpy()
    return masks

zs_masks_all = []
ft_masks_all = []

for i, (frames_dir, (pf, cx, cy)) in enumerate(zip(VID_FRAMES, prompts)):
    print(f'vid{i+1}  prompt_frame={pf}  click=({cx},{cy})')
    print(f'  zero-shot...', end=' ', flush=True)
    zs_masks_all.append(run_video_inference(zs_predictor, frames_dir, pf, cx, cy))
    print(f'{len(zs_masks_all[-1])} frames')
    print(f'  fine-tuned...', end=' ', flush=True)
    ft_masks_all.append(run_video_inference(ft_predictor, frames_dir, pf, cx, cy))
    print(f'{len(ft_masks_all[-1])} frames')

print(f'\nDone — {N_STUDIES} studies × 2 models')

## 8 — Frame-by-Frame Comparison Grid

In [ ]:
def add_label_bar(ax, text, color):
    ax.text(0.5, 0.97, text, transform=ax.transAxes,
            fontsize=8, fontweight='bold', color='white',
            ha='center', va='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85, linewidth=0))

def comparison_grid(frame_files, zs_masks, ft_masks, prompt_frame, title, save_path, n_cols=10):
    indices    = np.linspace(0, len(frame_files)-1, n_cols, dtype=int)
    row_colors = ['#333333', '#c0392b', '#2471a3']
    row_labels = ['Original', 'Zero-Shot MedSAM2', 'Fine-Tuned MedSAM2']

    fig, axes = plt.subplots(3, n_cols, figsize=(3.6*n_cols, 11))
    for col, idx in enumerate(indices):
        frame = np.array(Image.open(frame_files[idx]))
        zs    = zs_masks.get(idx, np.zeros(frame.shape[:2], dtype=bool))
        ft    = ft_masks.get(idx, np.zeros(frame.shape[:2], dtype=bool))
        star  = '★ ' if idx == prompt_frame else ''

        axes[0][col].imshow(frame, cmap='gray')
        axes[0][col].set_title(f'{star}Fr.{idx}', fontsize=9)
        axes[0][col].axis('off')

        axes[1][col].imshow(frame, cmap='gray')
        axes[1][col].imshow(zs, alpha=0.45, cmap='Reds')
        axes[1][col].axis('off')

        axes[2][col].imshow(frame, cmap='gray')
        axes[2][col].imshow(ft, alpha=0.45, cmap='Blues')
        axes[2][col].axis('off')

        for row, (color, label) in enumerate(zip(row_colors, row_labels)):
            add_label_bar(axes[row][col], label, color)

    for row, label in enumerate(row_labels):
        axes[row][0].set_ylabel(label, fontsize=11, rotation=90, labelpad=8)

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

for i, (frames, zs_masks, ft_masks, (pf, cx, cy)) in enumerate(
        zip(all_vid_frames, zs_masks_all, ft_masks_all, prompts)):
    comparison_grid(
        frames, zs_masks, ft_masks, pf,
        f'vid{i+1} — Zero-Shot vs Fine-Tuned MedSAM2  (★ = prompt frame)',
        f'{OUT_DIR}/vid{i+1}_frame_comparison.png'
    )

subprocess.run(
    ['gsutil','-m','cp'] +
    [f'{OUT_DIR}/vid{i+1}_frame_comparison.png' for i in range(N_STUDIES)] +
    [f'{BUCKET}/results/video_eval/'],
    check=True)
print(f'All {N_STUDIES} frame grids saved to GCS')

## 9 — Comparison Video

In [ ]:
import cv2

def draw_label(panel, text, color_bgr):
    font      = cv2.FONT_HERSHEY_DUPLEX
    scale     = 0.55
    thickness = 1
    pad       = 6
    (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
    x1, y1 = 8, 8
    x2, y2 = x1 + tw + pad*2, y1 + th + pad*2
    cv2.rectangle(panel, (x1, y1), (x2, y2), color_bgr, -1)
    cv2.rectangle(panel, (x1, y1), (x2, y2), (255,255,255), 1)
    cv2.putText(panel, text, (x1+pad, y2-pad), font, scale, (255,255,255), thickness, cv2.LINE_AA)

def draw_frame_counter(panel, frame_idx, total, prompt_frame):
    text = f'Fr {frame_idx}/{total-1}' + (' ★' if frame_idx == prompt_frame else '')
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.45
    (tw, th), _ = cv2.getTextSize(text, font, scale, 1)
    H, W = panel.shape[:2]
    cv2.putText(panel, text, (W - tw - 8, H - 8), font, scale, (200,200,200), 1, cv2.LINE_AA)

def make_comparison_video(frame_files, zs_masks, ft_masks, prompt_frame, out_path, fps=12):
    """3-panel MP4: Original | Zero-Shot MedSAM2 | Fine-Tuned MedSAM2, with labels."""
    sample = np.array(Image.open(frame_files[0]).convert('RGB'))
    H, W   = sample.shape[:2]
    div    = 2
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W*3+div*2, H))
    n      = len(frame_files)

    for i, path in enumerate(frame_files):
        frame = np.array(Image.open(path).convert('RGB'))
        zs    = zs_masks.get(i, np.zeros((H, W), dtype=bool))
        ft    = ft_masks.get(i, np.zeros((H, W), dtype=bool))

        orig     = frame.copy()
        zs_panel = frame.copy()
        zs_panel[zs] = (zs_panel[zs]*0.5 + np.array([220,40,40])*0.5).clip(0,255).astype(np.uint8)
        ft_panel = frame.copy()
        ft_panel[ft] = (ft_panel[ft]*0.5 + np.array([40,80,220])*0.5).clip(0,255).astype(np.uint8)

        orig_bgr = cv2.cvtColor(orig,     cv2.COLOR_RGB2BGR)
        zs_bgr   = cv2.cvtColor(zs_panel, cv2.COLOR_RGB2BGR)
        ft_bgr   = cv2.cvtColor(ft_panel, cv2.COLOR_RGB2BGR)

        draw_label(orig_bgr, 'Original',          (50,  50,  50))
        draw_label(zs_bgr,   'Zero-Shot MedSAM2', (30,  30, 180))
        draw_label(ft_bgr,   'Fine-Tuned MedSAM2',(160, 60,  20))

        for panel in [orig_bgr, zs_bgr, ft_bgr]:
            draw_frame_counter(panel, i, n, prompt_frame)

        divider  = np.full((H, div, 3), 200, dtype=np.uint8)
        writer.write(np.concatenate([orig_bgr, divider, zs_bgr, divider, ft_bgr], axis=1))

    writer.release()
    print(f'  Saved {out_path}  ({n} frames, {os.path.getsize(out_path)/1e6:.1f} MB)')

for i, (frames, zs_masks, ft_masks, (pf, cx, cy)) in enumerate(
        zip(all_vid_frames, zs_masks_all, ft_masks_all, prompts)):
    print(f'vid{i+1}...')
    make_comparison_video(
        frames, zs_masks, ft_masks, pf,
        f'{OUT_DIR}/vid{i+1}_comparison.mp4', fps=12
    )

subprocess.run(
    ['gsutil','-m','cp'] +
    [f'{OUT_DIR}/vid{i+1}_comparison.mp4' for i in range(N_STUDIES)] +
    [f'{BUCKET}/results/video_eval/'],
    check=True)
print(f'\nAll {N_STUDIES} videos at: {BUCKET}/results/video_eval/')